# **Поиск причинной структуры: PC, GES, LiNGAM и NO TEARS**

Практика к модулю [«Каузальность и линейные модели»](https://ai-interpretability.school), урок «Поиск причинной структуры».

В уроке четыре метода поиска структуры были формулами и задачами на три вершины. Здесь мы запустим их на данных, сгенерированных по известному графу из шести вершин, и посчитаем, сколько ребер каждый угадал. Проверим четыре утверждения урока:

1. PC и GES по наблюдениям восстанавливают граф только с точностью до класса марковской эквивалентности: часть ребер остается без направления;
2. LiNGAM ориентирует все ребра, если шумы негауссовы, и ошибается, если гауссовы;
3. NO TEARS находит граф на исходных данных и теряет его после стандартизации признаков;
4. на малой выборке PC и GES ошибаются, а при скрытом конфаундере PC добавляет ребра, которых нет.

План тетради:

1. граф и данные;
2. расстояние между графами (SHD);
3. PC;
4. GES;
5. LiNGAM;
6. NO TEARS;
7. где ломается.

Тетрадь идет на процессоре, прогон занимает около минуты.

In [ ]:
%pip install -q causal-learn==0.1.4.8

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import scipy.linalg as slin
import scipy.optimize as sopt
from causallearn.search.ConstraintBased.PC import pc
from causallearn.search.ScoreBased.GES import ges
from causallearn.search.FCMBased.lingam import DirectLiNGAM
from causallearn.graph.Dag import Dag
from causallearn.graph.GraphNode import GraphNode
from causallearn.utils.DAG2CPDAG import dag2cpdag

SEED = 0

## 1. Граф и данные

Шесть переменных связаны линейной моделью структурных уравнений $X_j = \sum_i B_{ij} X_i + e_j$:

$$X_1 \to X_2 \to X_4 \leftarrow X_3,\quad X_4 \to X_5 \to X_6 \leftarrow X_3.$$

Веса ребер подобраны так, чтобы связи по разным путям не гасили друг друга: иначе нарушится допущение верности (faithfulness), и независимость в данных появится там, где граф ее не обещает. Данные генерируем дважды: с гауссовым шумом $e_j \sim \mathcal N(0,1)$ и с равномерным на $[-\sqrt3, \sqrt3]$ той же дисперсии. Объектов 2000.

In [ ]:
names = ['X1', 'X2', 'X3', 'X4', 'X5', 'X6']
d = len(names)
edges = [(0, 1, 1.0), (1, 3, 1.2), (2, 3, -1.0), (3, 4, 0.8), (4, 5, 1.1), (2, 5, -0.9)]
B = np.zeros((d, d))
for i, j, w in edges:
    B[i, j] = w                      # B[i, j] != 0: ребро X_{i+1} -> X_{j+1}


def sample(n, noise, seed=SEED):
    rng = np.random.default_rng(seed)
    X = np.zeros((n, d))
    for j in range(d):               # вершины уже стоят в порядке графа
        e = rng.normal(0, 1, n) if noise == 'gauss' else rng.uniform(-np.sqrt(3), np.sqrt(3), n)
        X[:, j] = X @ B[:, j] + e
    return X


X_gauss, X_unif = sample(2000, 'gauss'), sample(2000, 'unif')
print('дисперсии, гауссов шум:   ', X_gauss.var(0).round(2))
print('дисперсии, равномерный шум:', X_unif.var(0).round(2))

## 2. Расстояние между графами

Граф будем хранить матрицей $A$: $A_{ij} = 1$ — ребро $i \to j$, $A_{ij} = A_{ji} = 1$ — ребро без направления. **SHD** (structural Hamming distance) — число пар вершин, у которых ребро отличается: лишнее, пропущенное, развернутое или без направления там, где оно есть. Ноль — графы совпадают.

PC и GES возвращают не один граф, а класс марковской эквивалентности — CPDAG, где ребра, направление которых данные не определяют, остаются без направления. Поэтому их выход сравниваем с CPDAG истинного графа, а выход LiNGAM и NO TEARS — с самим графом.

In [ ]:
A_true = (B != 0).astype(int)


def to_matrix(G):
    """Граф causal-learn -> матрица A: A[i, j] = 1 — ребро i -> j, A[i, j] = A[j, i] = 1 — без направления."""
    M, A = G.graph, np.zeros((d, d), int)
    for i in range(d):
        for j in range(d):
            if M[j, i] == 1 and M[i, j] == -1:
                A[i, j] = 1
            if M[i, j] == -1 and M[j, i] == -1:
                A[i, j] = A[j, i] = 1
    return A


def shd(A, A_hat):
    return sum((A[i, j], A[j, i]) != (A_hat[i, j], A_hat[j, i]) for i in range(d) for j in range(i + 1, d))


def show(A):
    out = []
    for i in range(d):
        for j in range(d):
            if A[i, j] and A[j, i] and i < j:
                out.append(f'{names[i]} - {names[j]}')
            elif A[i, j] and not A[j, i]:
                out.append(f'{names[i]} -> {names[j]}')
    return ', '.join(out)


nodes = [GraphNode(nm) for nm in names]
dag = Dag(nodes)
for i, j, _ in edges:
    dag.add_directed_edge(nodes[i], nodes[j])
A_cpdag = to_matrix(dag2cpdag(dag))
print('истинный граф:', show(A_true))
print('его CPDAG:    ', show(A_cpdag))
print('SHD(граф, CPDAG) =', shd(A_true, A_cpdag))

У ребра $X_1 \to X_2$ направление по наблюдениям не определить: граф с $X_2 \to X_1$ дает те же условные независимости. Остальные ребра ориентированы двумя коллайдерами, $X_2 \to X_4 \leftarrow X_3$ и $X_5 \to X_6 \leftarrow X_3$, и правилами Meek.

## 3. PC

PC ([Spirtes и Glymour, 1991](https://doi.org/10.1177/089443939100900106)) начинает с полного неориентированного графа и удаляет ребро между $X_i$ и $X_j$, как только находит набор $S$ соседей, при котором тест не отвергает условную независимость $X_i \perp X_j \mid S$. Для гауссовых данных тест — z-преобразование Фишера частной корреляции, `fisherz`; уровень значимости $\alpha = 0.05$. Затем ориентирует коллайдеры по найденным наборам $S$ и распространяет направления правилами Meek.

**Задание 1. Запустите PC на данных с гауссовым шумом (`pc(X_gauss, 0.05, 'fisherz')`), переведите результат в матрицу функцией `to_matrix` и посчитайте SHD до истинного графа `A_true` и до его CPDAG `A_cpdag`. В ответ — SHD до истинного графа.**

In [ ]:
# Ваш код здесь

PC нашел ровно класс эквивалентности: все ребра на месте, единственная ошибка до истинного графа — ребро $X_1 - X_2$ без направления, и это не ошибка метода, а предел того, что можно узнать из независимостей.

## 4. GES

GES ([Chickering, 2002](https://www.jmlr.org/papers/v3/chickering02b.html)) ищет не тестами, а оценкой: жадно добавляет ребра, пока растет BIC, потом жадно удаляет, и делает это в пространстве классов эквивалентности. Выход — тоже CPDAG.

In [ ]:
A_ges = to_matrix(ges(X_gauss, score_func='local_score_BIC')['G'])
print('GES:', show(A_ges))
print(f'SHD до истинного графа {shd(A_true, A_ges)} | до CPDAG {shd(A_cpdag, A_ges)}')

## 5. LiNGAM

LiNGAM ([Shimizu et al., 2006](https://www.jmlr.org/papers/v7/shimizu06a.html)) пользуется тем, что при негауссовом шуме регрессия в правильную сторону дает остаток, независимый от регрессора, а в неправильную — нет. DirectLiNGAM ([Shimizu et al., 2011](https://www.jmlr.org/papers/v12/shimizu11a.html)) по одной находит переменную, независимую от остатков всех регрессий на нее, ставит ее следующей в порядке и вычитает. Выход — граф с направлениями всех ребер и весами. Веса меньше 0.1 по модулю считаем нулевыми.

**Задание 2. Запустите `DirectLiNGAM` на данных с равномерным и с гауссовым шумом. Матрица весов — `model.adjacency_matrix_`, в ней строка — следствие, столбец — причина, поэтому ее надо транспонировать. Посчитайте SHD до истинного графа в обоих случаях. В ответ — SHD для гауссова шума.**

In [ ]:
# Ваш код здесь

С равномерным шумом LiNGAM нашел весь граф, включая направление $X_1 \to X_2$, которое PC и GES определить не могли. С гауссовым шумом остатки регрессии в обе стороны одинаково независимы от регрессора, и порядок, который выбирает DirectLiNGAM, перестает отражать граф.

## 6. NO TEARS

NO TEARS ([Zheng et al., 2018](https://arxiv.org/abs/1803.01422)) решает задачу

$$\min_W \frac{1}{2n}\|X - XW\|_F^2 + \lambda\|W\|_1 \quad \text{при} \quad h(W) = \operatorname{tr}\big(e^{W\circ W}\big) - d = 0$$

методом расширенного лагранжиана. Ниже — линейная версия из [репозитория авторов](https://github.com/xunzheng/notears) (лицензия Apache-2.0) почти без изменений; штраф $\lambda = 0.1$, веса меньше 0.3 по модулю обнуляются.

In [ ]:
def notears_linear(X, lambda1=0.1, max_iter=100, h_tol=1e-8, rho_max=1e16, w_threshold=0.3):
    """Линейный NO TEARS по notears/linear.py (Zheng et al., 2018; Apache-2.0), квадратичная потеря."""
    n, d = X.shape
    X = X - X.mean(axis=0, keepdims=True)

    def _h(W):
        E = slin.expm(W * W)
        return np.trace(E) - d, E.T * W * 2

    def _adj(w):
        return (w[:d * d] - w[d * d:]).reshape(d, d)

    def _func(w):
        W = _adj(w)
        R = X - X @ W
        loss, G_loss = 0.5 / n * (R ** 2).sum(), -1.0 / n * X.T @ R
        h, G_h = _h(W)
        obj = loss + 0.5 * rho * h * h + alpha * h + lambda1 * w.sum()
        G = G_loss + (rho * h + alpha) * G_h
        return obj, np.concatenate((G + lambda1, -G + lambda1), axis=None)

    w_est, rho, alpha, h = np.zeros(2 * d * d), 1.0, 0.0, np.inf
    bnds = [(0, 0) if i == j else (0, None) for _ in range(2) for i in range(d) for j in range(d)]
    for _ in range(max_iter):
        while rho < rho_max:
            w_new = sopt.minimize(_func, w_est, method='L-BFGS-B', jac=True, bounds=bnds).x
            h_new = _h(_adj(w_new))[0]
            if h_new > 0.25 * h:
                rho *= 10
            else:
                break
        w_est, h = w_new, h_new
        alpha += rho * h
        if h <= h_tol or rho >= rho_max:
            break
    W = _adj(w_est)
    W[np.abs(W) < w_threshold] = 0
    return W


A_nt = (notears_linear(X_gauss) != 0).astype(int)
print(f'NO TEARS на исходных данных: SHD {shd(A_true, A_nt)} | {show(A_nt)}')

На исходных данных с гауссовым шумом NO TEARS нашел весь граф с направлениями, хотя PC и GES направление $X_1 \to X_2$ определить не могли. Дело не в методе, а в данных: [Reisach, Seiler и Weichwald, 2021](https://arxiv.org/abs/2102.13647) заметили, что в таких симуляциях дисперсия растет вдоль графа — у следствия она больше, чем у причины, — и квадратичная потеря выбирает направление «от меньшей дисперсии к большей». Это свойство они назвали varsortability.

**Задание 3. Посчитайте на `X_gauss` долю ребер истинного графа, у которых дисперсия следствия больше дисперсии причины, — упрощенную версию varsortability. Ответ округлите до трех знаков после точки.**

In [ ]:
# Ваш код здесь

Дисперсия растет вдоль пяти ребер из шести, и NO TEARS этим пользуется.

**Задание 4. Стандартизуйте признаки `X_gauss` (вычтите среднее, разделите на стандартное отклонение), запустите `notears_linear` и посчитайте SHD до истинного графа.**

In [ ]:
# Ваш код здесь

После стандартизации подсказка исчезает, и NO TEARS ошибается в пяти парах вершин: четыре ребра развернуты, одно пропущено. PC и GES от масштаба не зависят: частные корреляции и BIC линейной гауссовой модели не меняются при умножении признака на число.

In [ ]:
X_std = (X_gauss - X_gauss.mean(0)) / X_gauss.std(0)
shd_pc_std = shd(A_cpdag, to_matrix(pc(X_std, 0.05, 'fisherz', show_progress=False).G))
shd_ges_std = shd(A_cpdag, to_matrix(ges(X_std, score_func='local_score_BIC')['G']))
print(f'PC: SHD {shd_pc_std} | GES: SHD {shd_ges_std}')

## 7. Где ломается

### 7.1. Мало данных

PC и GES на гауссовых данных разного размера, по пять случайных выборок на каждый размер; в таблице — средний SHD до CPDAG.

In [ ]:
rows = []
for n in (100, 200, 500, 1000, 2000):
    shd_pc, shd_ges = [], []
    for s in range(5):
        Xn = sample(n, 'gauss', seed=s)
        shd_pc.append(shd(A_cpdag, to_matrix(pc(Xn, 0.05, 'fisherz', show_progress=False).G)))
        shd_ges.append(shd(A_cpdag, to_matrix(ges(Xn, score_func='local_score_BIC')['G'])))
    rows.append(dict(n=n, PC=np.mean(shd_pc), GES=np.mean(shd_ges)))
pd.DataFrame(rows)

На 100–1000 объектах PC ошибается в среднем в 1.6–3.8 парах вершин: тест не отвергает слабую зависимость, ребро удаляется, а вслед за ним неверно ориентируются коллайдеры. GES на этих данных ошибается реже, но тоже не всегда находит класс эквивалентности.

### 7.2. Скрытый конфаундер

Уберем из данных $X_3$ — общую причину $X_4$ и $X_6$ — и запустим PC на оставшихся пяти переменных.

In [ ]:
keep = [0, 1, 3, 4, 5]
G_hidden = pc(X_gauss[:, keep], 0.05, 'fisherz', show_progress=False, node_names=[names[k] for k in keep]).G
print(G_hidden)

PC добавил ребра $X_4 \to X_6$ и $X_2 \to X_6$, которых в графе нет: без $X_3$ зависимость $X_4$ и $X_6$ ничем не объясняется, кроме прямого ребра. PC, GES, LiNGAM и NO TEARS в этой тетради предполагают, что все общие причины измерены. Для данных со скрытыми конфаундерами есть FCI — вариант PC, который помечает ребра, за которыми может стоять неизмеренная причина (`causallearn.search.ConstraintBased.FCI.fci`).

## Что унести из тетради

- **PC и GES находят класс эквивалентности, а не граф.** На 2000 объектов оба нашли CPDAG без ошибок, но ребро $X_1 - X_2$ направить не смогли — по независимостям его не направить никак.
- **LiNGAM направляет все ребра, если шум негауссов.** С равномерным шумом SHD 0, с гауссовым — 8.
- **NO TEARS на симуляциях выигрывает за счет масштаба.** Дисперсия растет вдоль пяти ребер из шести; после стандартизации SHD вырастает с 0 до 5, а PC и GES на стандартизованных данных не меняются.
- **Найденный граф — гипотеза.** На малой выборке ошибки идут каскадом, а скрытый конфаундер добавляет ребра, которых нет.

Источники:
1. [Spirtes & Glymour, 1991. An Algorithm for Fast Recovery of Sparse Causal Graphs](https://doi.org/10.1177/089443939100900106)
2. [Chickering, 2002. Optimal Structure Identification With Greedy Search](https://www.jmlr.org/papers/v3/chickering02b.html)
3. [Shimizu, Hoyer, Hyvärinen & Kerminen, 2006. A Linear Non-Gaussian Acyclic Model for Causal Discovery](https://www.jmlr.org/papers/v7/shimizu06a.html)
4. [Shimizu et al., 2011. DirectLiNGAM: A Direct Method for Learning a Linear Non-Gaussian Structural Equation Model](https://www.jmlr.org/papers/v12/shimizu11a.html)
5. [Zheng, Aragam, Ravikumar & Xing, 2018. DAGs with NO TEARS: Continuous Optimization for Structure Learning](https://arxiv.org/abs/1803.01422)
6. [Reisach, Seiler & Weichwald, 2021. Beware of the Simulated DAG! Causal Discovery Benchmarks May Be Easy To Game](https://arxiv.org/abs/2102.13647)
7. [Meek, 1995. Causal Inference and Causal Explanation with Background Knowledge](https://arxiv.org/abs/1302.4972)